In [3]:
import mitsuba as mi
import drjit as dr

import numpy as np
from math import ceil

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from copy import copy

import importlib
import sys
from pathlib import Path

from PIL import Image

import os

if os.name == 'nt':
    sys.path.append(os.getcwd() + "\\..\\src")    
else:
    sys.path.append(os.getcwd() + "/../src")

# print(sys.path)
# print(mi.variants())

# mi.set_variant("llvm_ad_rgb")

os.environ["OPENCV_IO_ENABLE_OPENEXR"]="1"
import cv2

mi.set_variant("cuda_ad_rgb")

import torch
import numpy as np
import tqdm
import time

import pickle

import normflows as nf

import tinycudann as tcnn
import pprint

from scipy.ndimage import gaussian_filter1d

import matplotlib.pyplot as plt
from dataclasses import dataclass

from coordinate_system import CoordinateSystem

import renderers
import scenes
import targets
import models
import priors
import normalizing_flows
import input_encodings
import kernels
import utils
import contexts

importlib.reload(renderers)
importlib.reload(scenes)
importlib.reload(targets)
importlib.reload(models)
importlib.reload(priors)
importlib.reload(normalizing_flows)
importlib.reload(input_encodings)
importlib.reload(kernels)
importlib.reload(utils)
importlib.reload(contexts)

import renderers as rr
import scenes
import targets
import models
import priors
import normalizing_flows
from input_encodings import (build_hashgrid_encoding_conf, 
                             build_identity_encoding_conf, 
                             build_one_blob_encoding_conf,
                             build_spherical_harmonics_encoding)
import utils
import kernels
from contexts import Context

In [12]:
@dataclass
class ExperimentSetup:
    # Setup Info
    coordinate_system: CoordinateSystem
    prior_str: str
    scene_name: str

    # NN info
    num_flows: int
    num_bins: int
    hidden_units: int
    hidden_layers: int

    # learning rate
    lr: float

    # Encoding Info
    do_normal_encoding: bool
    do_in_dir_encoding: bool
    do_pos_encoding: bool
    
    pos_is_hash_grid: bool
    encode_pos_each_flow: bool
    latent_to_hashgrid: bool
    do_scaling_grid_sizes: bool

    grid_resolutions: list[int]
    grid_levels: list[int]
    grid_scale_factors: list[float]
    hash_grid_log2_size: int = 19 # Need to do this because I am not sure I can automatically load older versions otherwise

@dataclass
class Experiment:
    setup: ExperimentSetup

    # Info from training
    trained_iterations: int
    sample_gen_strategy: str
    sample_model: bool
    losses: list[int]
    tracked_errors: bool
    tracked_error_every: int
    mape_list: list[int]
    smape_list: list[int]
    mae_list: list[int]
    rmse_list: list[int]
    error_calcs_clipped: bool
    model: models.NormalizingFlowModel | None
    model_path: str | None = None



def load_model(model, path):
    model.load_state_dict(torch.load(path, weights_only=False))


In [13]:
def get_encodings(input_dim, 
                  normal_context_size, 
                  in_dir_context_size,
                  pos_context_size, 
                  pos_is_hash_grid=True,
                  encode_normal_each_flow = False, 
                  encode_in_dir_each_flow = False,
                  encode_pos_each_flow = True,
                  hash_grid_log2_size: int = 19, 
                  latent_to_hash_enc = False):
    direction_encoding, latent_context_size = build_one_blob_encoding_conf(input_dim, 32)

    pos_hash_grid_size = pos_context_size
    if latent_to_hash_enc:
        pos_hash_grid_size += 1

    # normal_encoding, normal_enc_out = build_one_blob_encoding_conf(normal_context_size, 32) 

    # normal_encoding, normal_enc_out = (build_hashgrid_encoding_conf(normal_context_size, 16, 2) 
    #                                     if normal_context_size == 2 or normal_context_size == 3
    #                                     else build_one_blob_encoding_conf(normal_context_size, 16))
    
    normal_encoding, normal_enc_out = (build_spherical_harmonics_encoding(normal_context_size, 4) 
                                        if normal_context_size == 2 or normal_context_size == 3
                                        else build_one_blob_encoding_conf(normal_context_size, 8))

    # in_dir_encoding, in_dir_enc_out = build_one_blob_encoding_conf(in_dir_context_size, 16)

    in_dir_encoding, in_dir_enc_out = (build_spherical_harmonics_encoding(in_dir_context_size, 4) 
                                        if in_dir_context_size == 2 or in_dir_context_size == 3
                                        else build_one_blob_encoding_conf(in_dir_context_size, 8))

    if pos_is_hash_grid:
        positional_encoding, pos_enc_out = (build_hashgrid_encoding_conf(pos_hash_grid_size, levels=8, features_per_level=2, log2_hashmap_size=hash_grid_log2_size)                                         
                                            if pos_hash_grid_size == 2 or pos_hash_grid_size == 3 or pos_hash_grid_size == 4
                                            else build_one_blob_encoding_conf(pos_hash_grid_size, 16))
    else:
        positional_encoding, pos_enc_out = build_one_blob_encoding_conf(pos_hash_grid_size, 32)


    flow_normal_input = (normal_context_size if encode_normal_each_flow else normal_enc_out) if normal_context_size > 0 else 0
    flow_in_dir_input = (in_dir_context_size if encode_in_dir_each_flow else in_dir_enc_out) if in_dir_context_size > 0 else 0
    flow_pos_input =    (pos_context_size if encode_pos_each_flow else pos_enc_out)          if pos_context_size    > 0 else 0

    identity_normal_encoding, _ = build_identity_encoding_conf(flow_normal_input)
    identity_in_dir_encoding, _ = build_identity_encoding_conf(flow_in_dir_input)
    identity_position_encoding, _ = build_identity_encoding_conf(flow_pos_input)

    model_enc_list = []
    if normal_context_size > 0: model_enc_list.append(normal_encoding if not encode_normal_each_flow  else identity_normal_encoding)
    if in_dir_context_size > 0: model_enc_list.append(in_dir_encoding if not encode_in_dir_each_flow  else identity_in_dir_encoding)
    if pos_context_size > 0: model_enc_list.append(positional_encoding if not encode_pos_each_flow    else identity_position_encoding)

    model_enc = None
    model_encoding_config = None
    if len(model_enc_list) > 0:
        model_encoding_config = {
            "otype": "Composite",
            "nested": model_enc_list
        }
        model_encoding_input_size = normal_context_size + in_dir_context_size + pos_context_size

        model_enc = tcnn.Encoding(model_encoding_input_size, model_encoding_config)


    flow_enc_list = []
    
    if latent_context_size > 0: flow_enc_list.append(direction_encoding)
    if normal_context_size > 0: flow_enc_list.append(normal_encoding  if encode_normal_each_flow  else identity_normal_encoding)
    if in_dir_context_size > 0: flow_enc_list.append(in_dir_encoding  if encode_in_dir_each_flow  else identity_in_dir_encoding)
    if pos_context_size > 0: flow_enc_list.append(positional_encoding if encode_pos_each_flow     else identity_position_encoding)


    flow_encoding_config = {
        "otype": "Composite",
        "nested": flow_enc_list
    }

    model_context_out = model_enc.n_output_dims if model_enc else 0
    flow_enc = tcnn.Encoding(model_context_out + 1, flow_encoding_config)

    return model_encoding_config, flow_encoding_config, flow_normal_input + flow_in_dir_input + flow_pos_input


In [14]:
def get_default_prior(means, stds):
    return priors.DiagGaussianPrior(means, stds)

def build_Normflow_model(prior, 
                         domain,
                         coordinate_system,
                         scene: mi.Scene,
                         num_flows = 4,
                         num_bins = 8,
                         hidden_units = 32, 
                         hidden_layers = 2,
                         do_normal_encoding = True, 
                         do_in_dir_encoding = False, 
                         do_pos_encoding = False,
                         encode_normal_each_flow = False,
                         encode_in_dir_each_flow = False,
                         pos_is_hash_grid = True, 
                         encode_pos_each_flow = True,
                         hash_grid_log2_size = 19, 
                         latent_to_hash_enc = False,
                         scaling_grid_sizes = False,
                         grid_resolutions = None,
                         levels           = None,     
                         scale_factors    = None) -> models.AbstractModel:
    

    input_dim           = 1 if not latent_to_hash_enc else 0

    normal_context_size = 3 if do_normal_encoding else 0
    in_dir_context_size = 3 if do_in_dir_encoding else 0
    pos_context_size    = 3 if do_pos_encoding    else 0

    if latent_to_hash_enc and not encode_pos_each_flow:
        raise RuntimeError("If latent variable is hash encoded positional encoding needs to be encoded for each flow")

    model_enc_config, flow_enc_config, flow_context_size = get_encodings(input_dim, 
                                                                         normal_context_size, 
                                                                         in_dir_context_size,
                                                                         pos_context_size, 
                                                                         pos_is_hash_grid=pos_is_hash_grid, 
                                                                         encode_normal_each_flow=encode_normal_each_flow,
                                                                         encode_in_dir_each_flow=encode_in_dir_each_flow,
                                                                         encode_pos_each_flow=encode_pos_each_flow,
                                                                         hash_grid_log2_size=hash_grid_log2_size,
                                                                         latent_to_hash_enc=latent_to_hash_enc)

    pprint.pprint(model_enc_config)
    print("------")
    pprint.pprint(flow_enc_config)
    print("------")
    print(flow_context_size)


    net_config = {
        "otype": "FullyFusedMLP",    # Component type.
        "activation": "ReLU",        # Activation of hidden layers.
        "output_activation": "None", # Activation of the output layer.
        "n_neurons": hidden_units,            # Neurons in each hidden layer.
                                    # May only be 16, 32, 64, or 128.
        "n_hidden_layers": hidden_layers,        # Number of hidden layers.
    }
    

    flows = []
    # grid_resolutions = [4 , 4 , 8 , 8 , 16 , 16 , 32 , 32]
    if grid_resolutions is None:
        grid_resolutions = [16] * num_flows
    
    if levels is None:
        levels = [8] * num_flows

    if scale_factors is None:
        scale_factors = [2.0] * num_flows


    # grid_resolutions = [4 , 4 , 6 , 6 , 8 , 8 , 12 , 12]
    # levels           = [2  , 2  , 4  , 4  , 5  , 5  , 6  , 6]       
    # scale_factors     = [1.0, 1.0, 1.5, 1.5, 2.0, 2.0, 3.0, 3.0]

    # grid_resolutions = [16] * 8
    # levels           = [8] * 8     
    # scale_factor     = [2.0] * 8

    

    for i in range(num_flows):
        if do_pos_encoding and scaling_grid_sizes:

            # Compute index where the hash grid encoding is
            pos_encodings_index = 1 if do_normal_encoding else 0
            if not latent_to_hash_enc:
                pos_encodings_index += 1

            if do_in_dir_encoding:
                pos_encodings_index += 1

            flow_enc_config['nested'][pos_encodings_index]['n_levels']        = levels[i]
            flow_enc_config['nested'][pos_encodings_index]['per_level_scale'] = scale_factors[i]
            flow_enc_config['nested'][pos_encodings_index]['base_resolution'] = grid_resolutions[i]

        flows += [normalizing_flows.NeuralSplineFlow(num_bins, flow_context_size, flow_enc_config, net_config, latent_to_hash_grid = latent_to_hash_enc)]

    

    return models.NormalizingFlowModel(prior, 
                                       flows, 
                                       normal_context_size + in_dir_context_size + pos_context_size,
                                       coordinate_system,
                                       scene_bbox = scene.bbox(),
                                       has_normal_encoding = do_normal_encoding,
                                       has_incoming_encoding = do_in_dir_encoding,
                                       has_positional_encoding = do_pos_encoding,
                                       context_encoding_conf = model_enc_config,
                                       domain=domain)

def buildReparamModel(features, 
                    prior, 
                    encoding,
                    domain,
                    hidden_layers = 2, 
                    hidden_units = 16):
    hidden_layers = [hidden_units] * (hidden_layers + 1)

    return models.ReparameterizationModel(features, prior, encoding, hidden_layers, domain=domain)

def buildNaiveModel(features, 
                    prior, 
                    encoding,
                    domain,
                    hidden_layers = 2, 
                    hidden_units = 16):
    hidden_layers = [hidden_units] * (hidden_layers + 1)

    return models.NaiveForwardModel(features, prior, encoding, hidden_layers, domain=domain)

def sample_normals(num_samples, device):
    domain = torch.tensor([[2 * torch.pi, torch.pi]], device = device)
    phi_theta = torch.rand((num_samples, 2), device = device) * domain
    return utils.to_vec3_torch(phi_theta[..., 0], phi_theta[..., 1])

def sample_random_dirs(num_samples, device):
    pos = torch.randn((num_samples, 3), device = device, dtype=torch.float32)
    return torch.nn.functional.normalize(pos, dim = -1)


def compute_reference(renderer, w, h, ref_spp, bounces = 4):
    rCtx = Context(bounces=bounces, spp=ref_spp, bsdf_sampling=True, emitter_sampling=True, mis_beta=2)
    return renderer.render(w, h, rCtx)

def compute_error_on_images(image, ref, metric = 'MAPE'):
    eps = 0.01
    if metric is 'MAPE':
        return np.mean(np.abs(image - ref) / (ref + eps), axis = None) 
    elif metric is 'SMAPE':
        error_image_smape = np.abs(image - ref) / (np.abs(ref) + np.abs(image) + eps)
        return 2 * np.mean(error_image_smape, axis = None) 
    elif metric is 'MAE':
        return np.mean(np.abs(image - ref), axis = None)
    elif metric is 'RMSE':
        return np.sqrt(np.mean(np.square(image - ref), axis = None))
    else:
        raise RuntimeError(f"Not a valid error metric: {metric}")

def compute_error_metric(renderer, 
                model, 
                w, 
                h, 
                ref_image = None, 
                ref_spp = 128, 
                model_spp = 1, 
                plot = True,
                num_test_images = 1, 
                clamp = False, 
                tonemap = False, 
                show_error_image = True,
                display_error_metric = 'MAPE', # Can currently compute MAPE, SMAPE, MAE, MRSE
                ref_image_label = None):
    rCtx = Context(bounces=4, spp=ref_spp, bsdf_sampling=True, emitter_sampling=True, mis_beta=2)
    
    rCtx.spp = ref_spp
    was_none = False
    if ref_image is None:
        was_none = True
        ref_image = compute_reference(renderer, w, h, ref_spp)
    
    
    rCtx.spp = model_spp
    MAPE =  0
    SMAPE = 0
    MAE =   0
    RMSE =  0

    for i in range(num_test_images):
        model_res_dr = renderer.render_with_model(w, h, model, rCtx)
        if clamp:
            model_res_dr = dr.clip(model_res_dr, 0.0, 1.0)
        
        eps = 0.01
        abs_diff = dr.abs(model_res_dr - ref_image)

        error_image_mape = abs_diff / (ref_image + eps)
        MAPE += dr.mean(error_image_mape, axis = None) 
            
        error_image_smape = abs_diff / (dr.abs(ref_image) + dr.abs(model_res_dr) + eps)
        SMAPE += 2 * dr.mean(error_image_smape, axis = None) 

        error_image_mae = abs_diff
        MAE += dr.mean(error_image_mae, axis = None)
        
        error_image_rmse = dr.square(model_res_dr - ref_image)
        RMSE += dr.sqrt(dr.mean(error_image_rmse, axis = None))
        
    
    mean_MAPE  = MAPE.item()  / num_test_images
    mean_SMAPE = SMAPE.item() / num_test_images
    mean_MAE   = MAE.item()   / num_test_images
    mean_RMSE  = RMSE.item()  / num_test_images

    error_dict = {
        'MAPE':  mean_MAPE,
        'SMAPE': mean_SMAPE,
        'MAE':   mean_MAE,
        'RMSE':  mean_RMSE
    }

    if not plot:
        return error_dict

    
    model_res     = utils.res_to_numpy_3(model_res_dr, w, h)
    res_nee       = utils.res_to_numpy_3(ref_image, w, h)
    
    if tonemap:
        model_res = utils.tonemap(model_res)
        res_nee   = utils.tonemap(res_nee)

    fig, axs = plt.subplots(1, 3, figsize = (15, 5)) if show_error_image else plt.subplots(1, 2, figsize = (10, 5))

    axs[0].imshow(model_res)
    axs[0].set_title(f"model path guided ({model_spp} spp)")

    ref_caption = f"reference ({ref_spp} spp)" if was_none else "reference"
    if ref_image_label is not None:
        ref_caption = f"{ref_image_label} ({ref_spp} spp)"

    axs[1].imshow(res_nee)
    axs[1].set_title(ref_caption)

    if show_error_image:
        error_image = None
        if display_error_metric == 'MAPE':
            error_image = error_image_mape
        elif display_error_metric == 'SMAPE':
            error_image = error_image_smape
        elif display_error_metric == 'MAE':
            error_image = error_image_mae
        elif display_error_metric == 'RMSE':
            error_image = error_image_rmse
        else:
            raise RuntimeError("Not a valid error metric given for pixelwise display")

        err_image_np  = utils.res_to_numpy_3(error_image, w, h)

        axs[2].imshow(err_image_np)
        axs[2].set_title(f"pixelwise error ({display_error_metric})")

    return fig, error_dict

<>:152: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:154: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:157: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:159: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
C:\Users\janlu\AppData\Local\Temp\ipykernel_6504\3430715052.py:152: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if metric is 'MAPE':
C:\Users\janlu\AppData\Local\Temp\ipykernel_6504\3430715052.py:154: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif metric is 'SMAPE':
C:\Users\janlu\AppData\Local\Temp\ipykernel_6504\3430715052.py:157: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif metric is 'MAE':
C:\Users\janlu\AppData\Local\Temp\ipykernel_6504\3430715052.py:159: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif metric is 'RMSE':


In [15]:
def create_model(
        coordinate_system,
        prior_str,
        scene,
        do_normal_encoding = True,
        do_in_dir_encoding = True,
        do_pos_encoding    = True,
        num_flows = 8,
        num_bins = 8,
        hidden_units = 64, 
        hidden_layers = 2,
        pos_is_hash_grid = True,
        encode_pos_each_flow = True,
        latent_to_hash_enc = True,
        hash_grid_log2_size = 19,
        scaling_grid_sizes = False):
    
    
    enable_cuda = True
    device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')

    # coordinate_system = CoordinateSystem.Cylindrical
    domain = utils.get_domain_from_coords(coordinate_system, device)


    if prior_str == 'cosine':
        prior = priors.CosineWeighted(coordinate_system)
    elif prior_str == 'uniform':
        prior = priors.UniformPrior(domain, coordinate_system)
    else:
        raise RuntimeError(f"{prior_str} is not a valid prior string.")
    
    model: models.AbstractModel = build_Normflow_model(prior, 
                                                    domain, 
                                                    coordinate_system,
                                                    scene=scene,
                                                    num_flows = num_flows, 
                                                    num_bins = num_bins, 
                                                    hidden_units=hidden_units, 
                                                    hidden_layers=hidden_layers,
                                                    do_normal_encoding=do_normal_encoding,
                                                    do_in_dir_encoding=do_in_dir_encoding,
                                                    do_pos_encoding=do_pos_encoding,
                                                    pos_is_hash_grid = pos_is_hash_grid,
                                                    encode_pos_each_flow = encode_pos_each_flow,
                                                    hash_grid_log2_size = hash_grid_log2_size,
                                                    latent_to_hash_enc   = latent_to_hash_enc,
                                                    scaling_grid_sizes   = scaling_grid_sizes,
                                                    grid_resolutions = None,
                                                    levels           = None,     
                                                    scale_factors    = None)



    model.to(device)
    print(f"using device: {device}")

    # fig = model.visualize(renderer)
    # plt.show()

    return model


def create_model_from_Setup(setup: ExperimentSetup):
    enable_cuda = True
    device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')

    # coordinate_system = CoordinateSystem.Cylindrical
    domain = utils.get_domain_from_coords(setup.coordinate_system, device)


    if setup.prior_str == 'cosine':
        prior = priors.CosineWeighted(setup.coordinate_system)
    elif setup.prior_str == 'uniform':
        prior = priors.UniformPrior(domain, setup.coordinate_system)
    else:
        raise RuntimeError(f"{setup.prior_str} is not a valid prior string.")
    
    scene = scenes.get_scene_by_name(setup.scene_name)

    model: models.AbstractModel = build_Normflow_model(prior, 
                                                    domain, 
                                                    setup.coordinate_system,
                                                    scene=scene,
                                                    num_flows = setup.num_flows, 
                                                    num_bins = setup.num_bins, 
                                                    hidden_units=setup.hidden_units, 
                                                    hidden_layers=setup.hidden_layers,
                                                    do_normal_encoding=setup.do_normal_encoding,
                                                    do_in_dir_encoding=setup.do_in_dir_encoding,
                                                    do_pos_encoding=setup.do_pos_encoding,
                                                    pos_is_hash_grid = setup.pos_is_hash_grid,
                                                    encode_pos_each_flow = setup.encode_pos_each_flow,
                                                    hash_grid_log2_size = setup.hash_grid_log2_size,
                                                    latent_to_hash_enc   = setup.latent_to_hashgrid,
                                                    scaling_grid_sizes   = setup.do_scaling_grid_sizes,
                                                    grid_resolutions = setup.grid_resolutions,
                                                    levels           = setup.grid_levels,     
                                                    scale_factors    = setup.grid_scale_factors)



    model.to(device)
    print(f"using device: {device}")

    # fig = model.visualize(renderer)
    # plt.show()

    return model

In [16]:
def load_Experiment(file_name: str):
    path = '../../data/experiments/' + file_name
    with open(path, 'rb') as inp:
        exp: Experiment = pickle.load(inp)
    
    if exp.model_path is not None:
        model = create_model_from_Setup(exp.setup)
        load_model(model, exp.model_path)
        exp.model = model

    return exp

In [17]:
exp = load_Experiment("hashgrid_setups_single_50k.pkl")

{'nested': [{'degree': 4, 'n_dims_to_encode': 3, 'otype': 'SphericalHarmonics'},
            {'degree': 4, 'n_dims_to_encode': 3, 'otype': 'SphericalHarmonics'},
            {'base_resolution': 16,
             'interpolation': 'Linear',
             'log2_hashmap_size': 19,
             'n_dims_to_encode': 3,
             'n_features_per_level': 2,
             'n_levels': 8,
             'otype': 'Grid',
             'per_level_scale': 2.0,
             'type': 'Hash'}],
 'otype': 'Composite'}
------
{'nested': [{'n_bins': 32, 'n_dims_to_encode': 1, 'otype': 'OneBlob'},
            {'n_dims_to_encode': 16, 'otype': 'Identity'},
            {'n_dims_to_encode': 16, 'otype': 'Identity'},
            {'n_dims_to_encode': 16, 'otype': 'Identity'}],
 'otype': 'Composite'}
------
48
using device: cuda


In [11]:
print(exp)

Experiment(setup=ExperimentSetup(coordinate_system=<CoordinateSystem.Spherical: (1,)>, prior_str='cosine', scene_name='cbox_inverted_light', num_flows=4, num_bins=16, hidden_units=64, hidden_layers=4, lr=0.0003, do_normal_encoding=True, do_in_dir_encoding=True, do_pos_encoding=True, pos_is_hash_grid=True, encode_pos_each_flow=False, latent_to_hashgrid=False, do_scaling_grid_sizes=False, grid_resolutions=[], grid_levels=[], grid_scale_factors=[], hash_grid_log2_size=19), trained_iterations=50000, sample_gen_strategy='light_tracing', sample_model=True, losses=array([150.31182861,  84.23020935,  74.80266571, ...,   8.81163692,
         8.88409042,   9.18109703], shape=(50000,)), tracked_errors=True, tracked_error_every=250, mape_list=array([0.86364764, 0.9695949 , 0.95907068, 0.91393232, 0.89519501,
       0.86485195, 0.82843608, 0.8493017 , 0.80809087, 0.83335924,
       0.75287056, 0.74273956, 0.72862166, 0.76756352, 0.70702112,
       0.70842791, 0.76620102, 0.69965237, 0.75343633, 0.6